# 9주차 ② LoRA 어댑터 학습 — 실습 4~5  〔빈칸본〕

> **빈칸이 2곳입니다.** 둘 다 셀 5(학습 루프)에 있고, **7주차 3교시 AMP 와 글자 그대로 같습니다.**
> LoRA 에서 바뀌는 것은 모델을 `get_peft_model` 로 감싸는 것 하나뿐이라는 걸
> 손으로 확인하는 것이 이 빈칸의 목적입니다.
> 다 채운 노트북은 `15_lora_finetune.ipynb` 로 저장해 제출합니다.

**목표**: `ΔW ≈ B·A` 라는 LoRA 의 원리를 **파라미터 수 계산으로** 확인하고,
`peft` 로 어댑터를 붙여 학습시킨 뒤 **어댑터 파일이 얼마나 작은지** 직접 본다.

> **과제 제출 대상 노트북입니다.**

```
   백본 동결   : 학습 0.01%   싸다   그런데 백본이 내 데이터에 맞춰지지 않는다
   전체 미세조정 : 학습 100%    비싸다  8,600만 개를 다 갱신 = 옵티마이저 상태까지 메모리 3배

                             중간은 없나?
```

미세조정이란 결국 **가중치 `W` 를 `W + ΔW` 로 바꾸는 것**입니다.
그럼 `ΔW` 만 따로 학습하면? — `ΔW` 도 `W` 와 크기가 같으니 그대로는 이득이 없습니다.
**여기서 한 가지 관찰이 들어갑니다.**

```
                ΔW                    ≈           B      ×      A
        ┌─────────────────┐              ┌───┐        ┌─────────────────┐
        │                 │              │   │        │                 │
        │   768 × 768     │      ≈       │768│   ×    │   r × 768       │
        │   = 589,824 개  │              │× r│        │                 │
        └─────────────────┘              └───┘        └─────────────────┘

        r = 8 이면   768×8 + 8×768 = 12,288 개
                     589,824 → 12,288        약 2% ★
```

> **핵심 메시지 ★★ (기말 출제 1순위)**: **`ΔW ≈ B · A`**
> - `A` : 768 → **r** 로 줄이는 행렬
> - `B` : **r** → 768 로 되돌리는 행렬
> - **`r` 이 작을수록 학습 파라미터가 적다.** `r=8` 이면 약 2%.
>
> 원래 `W` 는 **그대로 얼려 두고**, `B·A` 만 학습합니다.
> 추론할 때는 `W + B·A` 로 합쳐서 씁니다.

| 이점 | 설명 |
|---|---|
| **학습 파라미터 급감** | 전체의 **0.1~1%**. 옵티마이저 상태 메모리도 같이 준다 |
| **성능은 근접** | 전체 미세조정에 가까운 정확도가 나온다 |
| **어댑터가 작다** ★ | 전체 모델 330MB vs **어댑터 3MB**. 태스크마다 하나씩 갈아 끼운다 |

```
   [실무에서의 그림]
     백본 330MB  (하나만 두고)
        ├── 어댑터_고양이강아지.bin   3MB
        ├── 어댑터_의료영상.bin       3MB
        └── 어댑터_불량품검사.bin     3MB      ← 태스크 전환이 3MB 로딩으로 끝난다
```

## 실습 4 — LoRA 적용 + 학습 파라미터 수 확인 ★★

In [ ]:
# 셀 0 — 1교시에서 이어서 (커널을 재시작했다면 이 셀부터)
import torch, torch.nn as nn, time, os, json
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.transforms import v2

DATA = "mydata"
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG = 224
MEAN, STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

train_tf = v2.Compose([v2.RandomResizedCrop(IMG, scale=(0.8, 1.0)), v2.RandomHorizontalFlip(),
                       v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])
val_tf   = v2.Compose([v2.Resize((IMG, IMG)),
                       v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])

train_set = ImageFolder(f"{DATA}/train", transform=train_tf)
val_set   = ImageFolder(f"{DATA}/val",   transform=val_tf)
CLASSES = train_set.classes
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=16)
print("클래스 :", CLASSES, "| 장치 :", device)

In [ ]:
# 셀 1 — 모델을 새로 로드하고 LoRA 를 건다
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model, TaskType

base = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(CLASSES),
    ignore_mismatched_sizes=True,
).to(device)

config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=8,                                  # ★ 저차원의 크기
    lora_alpha=16,                        # ★ 보통 r 의 2배
    lora_dropout=0.1,
    target_modules=["query", "value"],    # ★ 어텐션의 Q, V 에 건다
    modules_to_save=["classifier"],       # ★ 헤드는 통째로 학습 (새로 만든 층이므로)
)

model = get_peft_model(base, config).to(device)
model.print_trainable_parameters()        # ★ 이 한 줄이 오늘의 하이라이트

> **관찰 포인트 ★★**: 출력이 이렇게 나옵니다.
> ```
>   trainable params: xxx,xxx || all params: 86,xxx,xxx || trainable%: 0.xx
> ```
> **전체의 1% 미만**입니다. 8,600만 개 중 수십만 개만 학습합니다.

> **기말 C 구분 예상 문항**: *"`trainable%` 가 0.5% 로 나온 이유를 설명하시오."*
> → `r=8` 로 `ΔW` 를 근사했기 때문. `768×768` 대신 `768×8 + 8×768` 만 학습.

> **함정 ★**: `trainable%` 가 **0.00%** 이거나 헤드 파라미터 수와 같다면
> **`target_modules` 이름이 틀린 것**입니다. 어댑터가 하나도 안 붙었습니다.
> 모델마다 층 이름이 다릅니다 — ViT 는 `query`/`value`, 다른 모델은 `q_proj`/`v_proj` 인 경우가 많습니다.

In [ ]:
# 셀 2 — 이름이 맞는지 확인하는 법 (0.00% 가 나오면 여기부터)
names = sorted({n.split(".")[-1] for n, _ in base.named_modules() if n})
print("모델 안의 층 이름들 (일부) :")
print("  ", [n for n in names if n in ("query", "key", "value", "dense", "classifier")])

In [ ]:
# 셀 3 — 숫자가 맞는지 손으로 검산
r = 8
d = 768                      # ViT-base 의 hidden size
층수 = 12                    # 트랜스포머 블록 수
대상 = 2                     # query, value

lora_params = 층수 * 대상 * (d*r + r*d)
print(f"LoRA 파라미터 손계산 : {lora_params:,}")
print(f"전체 미세조정이라면  : {층수 * 대상 * d * d:,}")
print(f"→ 약 {층수*대상*d*d / lora_params:.0f}배 적다")

print("\nr 을 바꾸면")
for rr in [4, 8, 16, 32]:
    print(f"  r={rr:2d} : {층수*대상*2*d*rr:>9,d} 개")

> **핵심 메시지 ★★**: `d×r + r×d = 2dr` 이므로 학습 파라미터는 **`r` 에 비례**합니다.
> `r` 을 8에서 16으로 올리면 파라미터가 **2배**가 되고, 표현력은 늘지만 학습이 무거워집니다.

In [ ]:
# 셀 4 — 학습되는 층 이름만 출력
for n, p in model.named_parameters():
    if p.requires_grad:
        print(f"  {n:70s} {tuple(p.shape)}")

> **관찰 포인트**: `lora_A`, `lora_B` 라는 이름이 붙은 작은 행렬들과 `classifier` 만 나옵니다.
> **원래 가중치는 전부 얼어 있습니다.** 위 그림이 코드로 확인되는 순간입니다.

## 실습 5 — LoRA 어댑터 학습 ★★

In [ ]:
# 셀 5 — 학습 (루프는 1교시와 똑같다)
from torch.amp import autocast, GradScaler

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-4)   # ★ LoRA 는 lr 을 좀 작게
scaler = GradScaler(device, enabled=(device == "cuda"))

EPOCHS = 5

def evaluate(m):
    m.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            with autocast(device, enabled=(device == "cuda")):
                out = m(xb).logits
            c += (out.argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t

if device == "cuda": torch.cuda.reset_peak_memory_stats()
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()

        # ───── 빈칸 ① : autocast 로 감싸고 순전파·손실 (두 줄) ─────
        # 힌트:  with autocast(device, enabled=(device == "cuda")):
        #            loss = loss_fn(model(xb).______, yb)


        # ───── 빈칸 ② : scaler 로 역전파·갱신 (세 줄) ─────
        # 힌트:  7주차 3교시 AMP 그대로.  scale → step → update



    print(f"epoch {epoch+1}/{EPOCHS} | 검증 정확도 {evaluate(model)*100:5.2f}%")

lora_result = dict(
    방식="LoRA (r=8)",
    학습파라미터=sum(p.numel() for p in model.parameters() if p.requires_grad),
    정확도=evaluate(model),
    시간=time.time()-t0,
    VRAM=torch.cuda.max_memory_allocated()/1024**3 if device=="cuda" else 0,
)
print("\nLoRA :", lora_result)

> **핵심 메시지 ★**: **학습 루프가 1교시와 똑같습니다.** 7주차 AMP 도 그대로입니다.
> 바뀐 건 **모델을 `get_peft_model` 로 감쌌다는 것 하나**뿐입니다.
> LoRA 는 학습 방법이 아니라 **모델을 바꾸는 기법**입니다.

> **lr 이 다릅니다**: 백본 동결은 `1e-3`, LoRA 는 `1e-4` 를 썼습니다.
> LoRA 는 **원래 가중치 옆에 붙는 작은 보정**이라 큰 lr 을 주면 흔들립니다.
> 6주차에 *"옵티마이저를 바꾸면 lr 도 바꾼다"* 고 한 것과 같은 감각입니다.

In [ ]:
# 셀 6 — 어댑터만 저장하고 크기를 확인한다 ★
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
model.save_pretrained("models/lora_adapter")      # ★ 어댑터만 저장된다

def folder_mb(path):
    return sum(os.path.getsize(os.path.join(root, f))
               for root, _, fs in os.walk(path) for f in fs) / 1024**2

adapter_mb = folder_mb("models/lora_adapter")
full_mb = sum(p.numel() for p in base.parameters()) * 4 / 1024**2

print(f"어댑터 폴더  : {adapter_mb:8.2f} MB")
print(f"전체 ViT 모델: {full_mb:8.2f} MB (fp32 기준)")
print(f"→ {full_mb/adapter_mb:.0f}배 차이")

with open("results/lora.json", "w", encoding="utf-8") as f:
    json.dump({**lora_result, "어댑터MB": adapter_mb, "전체모델MB": full_mb},
              f, ensure_ascii=False, indent=2)
print("\nresults/lora.json 저장  ← 3교시 비교표에서 읽는다")

> **관찰 포인트 ★★**: **어댑터 수 MB vs 전체 모델 330MB.**
> 이 대비가 **LoRA 의 가치를 설명하는 가장 빠른 방법**입니다.
> 탐색기로 `models/lora_adapter` 폴더를 직접 열어 보세요.
> **과제에 이 두 숫자를 나란히 적으라**고 되어 있습니다.

In [ ]:
# 셀 7 (참고) — 다음에 어댑터를 불러 쓸 때 (실행할 필요 없음)
# from peft import PeftModel
# base  = ViTForImageClassification.from_pretrained(
#     "google/vit-base-patch16-224", num_labels=len(CLASSES), ignore_mismatched_sizes=True)
# model = PeftModel.from_pretrained(base, "models/lora_adapter")
print("위 세 줄이면 어댑터를 되살릴 수 있습니다. 백본은 매번 새로 받아도 되고요.")

> **막히면**:
> | 증상 | 원인 |
> |---|---|
> | `trainable%` 가 0.00% | `target_modules` 이름이 틀렸다 ★ (셀 2로) |
> | 정확도가 안 오른다 | lr 이 너무 작다. `1e-4` → `3e-4` 로 |
> | `CUDA out of memory` | batch 16 → 8 |
> | `.logits` 오류 | 1교시의 그 함정 |
> | 어댑터 폴더가 300MB | `modules_to_save` 에 너무 많이 넣었다 |

## `r` 의 감각

| `r` | 학습 파라미터 | 표현력 | 언제 |
|---|---|---|---|
| **4** | 아주 적음 | 낮음 | 데이터가 매우 적을 때 |
| **8** | 적음 | 보통 | **기본값** ★ 오늘 쓴 값 |
| **16~32** | 많음 | 높음 | 원래 문제와 많이 다른 데이터 |

```
   alpha / r  =  어댑터의 "세기"
     보통  alpha = 2 × r   로 둔다  (오늘: r=8, alpha=16)
```

> **핵심 메시지 ★**: `r` 을 무작정 키운다고 좋아지지 않습니다.
> **데이터가 200장인데 `r=32` 면 과적합**합니다 — 6주차에 배운 그것입니다.
> 과제에서 **`r` 을 바꿔 본 실험 1건**을 넣으면 좋은 보고서가 됩니다.

---

### 이 노트북 체크리스트

- [ ] `ΔW ≈ B·A` 를 그림으로 설명할 수 있다 ★★
- [ ] `r=8` 일 때 학습 파라미터가 왜 2% 정도인지 **계산으로** 보일 수 있다 ★
- [ ] `print_trainable_parameters()` 로 1% 미만을 확인했다
- [ ] `trainable%` 가 0.00% 면 무엇을 의심해야 하는지 안다
- [ ] LoRA 어댑터를 학습시켰다
- [ ] **어댑터 폴더 크기와 전체 모델 크기를 대비**해 봤다 ★
- [ ] `r` 을 키울 때의 득실을 말할 수 있다
- [ ] 기준선과 LoRA 의 정확도·VRAM·시간을 기록했다